# Решения · handbook гл. 35

Открывай **только после** своих попыток в [handbook_gl35_практика.ipynb](./handbook_gl35_практика.ipynb).

Источник: [backpropagation](https://mlhandbook-production.up.railway.app/parts/deep-learning/backprop.html)

Теория: [d.4.35 … .md](../d.4.35%20Backpropagation/d.4.35%20Backpropagation.md) · [README.md](./README.md)


## Зачем практику handbook

Закрепить главу **35** руками: каждый блок — мини-эксперимент из [ML Handbook](https://mlhandbook-production.up.railway.app/). Решения здесь — эталон для сверки после своего [handbook_gl35_практика.ipynb](./handbook_gl35_практика.ipynb).


# §35.3 · XOR NumPy

## Зачем этот фрагмент

XOR **не линейно разделим** — нужен скрытый слой; здесь backprop руками в NumPy по MSE + сигмоида.


In [1]:
import numpy as np


def need(cond, msg=""):
    assert cond, msg


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


rng = np.random.default_rng(0)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y = np.array([[0], [1], [1], [0]], float)
W1 = rng.normal(0, 1, (2, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 1, (4, 1))
b2 = np.zeros((1, 1))
lr = 0.5
n = X.shape[0]

for epoch in range(5000):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    d_a2 = 2 * (a2 - y) / n
    d_z2 = d_a2 * a2 * (1 - a2)
    d_W2 = a1.T @ d_z2
    d_b2 = np.sum(d_z2, axis=0, keepdims=True)
    d_a1 = d_z2 @ W2.T
    d_z1 = d_a1 * a1 * (1 - a1)
    d_W1 = X.T @ d_z1
    d_b1 = np.sum(d_z1, axis=0, keepdims=True)
    W2 -= lr * d_W2
    b2 -= lr * d_b2
    W1 -= lr * d_W1
    b1 -= lr * d_b1

pred = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2).round(2).ravel()
print("предсказания XOR:", pred)
need(np.allclose(pred, [0.02, 0.95, 0.95, 0.06], atol=0.05))


предсказания XOR: [0.02 0.95 0.95 0.06]


# §35.4 · XOR PyTorch

## Зачем этот фрагмент

Тот же XOR, но **autograd**: `loss.backward()` вместо ручных `d_W`. См. [d.4.35 …](../d.4.35%20Backpropagation/d.4.35%20Backpropagation.md).


In [2]:
import numpy as np
import torch
import torch.nn as nn


def need(cond, msg=""):
    assert cond, msg


X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y = np.array([[0], [1], [1], [0]], float)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32)

net = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1), nn.Sigmoid())
opt = torch.optim.Adam(net.parameters(), lr=0.1)
lossf = nn.MSELoss()

for epoch in range(2000):
    opt.zero_grad()
    loss = lossf(net(Xt), yt)
    loss.backward()
    opt.step()

out = net(Xt).detach().round().ravel().tolist()
print("предсказания XOR (torch):", out)
need(out == [0.0, 1.0, 1.0, 0.0])


предсказания XOR (torch): [0.0, 1.0, 1.0, 0.0]


# §35.6 · gradient check

## Зачем этот фрагмент

**Gradient check**: численная производная ≈ аналитическая — страховка, что backprop написан верно.


In [3]:
import numpy as np


def need(cond, msg=""):
    assert cond, msg


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def loss(w, x=2.0, y=1.0):
    a = sigmoid(w * x)
    return (a - y) ** 2


w = 0.5
h = 1e-5
num_grad = (loss(w + h) - loss(w - h)) / (2 * h)
a = sigmoid(w * 2.0)
ana_grad = 2 * (a - 1.0) * a * (1 - a) * 2.0
print(f"численный: {num_grad:.6f}, аналитический: {ana_grad:.6f}")
need(np.isclose(num_grad, ana_grad))
need(abs(num_grad - (-0.211508)) < 1e-4)


численный: -0.211508, аналитический: -0.211508
